<a href="https://colab.research.google.com/github/Fares-pr0g/ML-journey-ep-2-Experimenting-with-NN-s-in-PyTorch/blob/main/Learning_PyTorch_07_RNN's.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project one – predicting the sentiment of IMDb movie reviews

In [1]:
!pip install -q datasets

from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [2]:
imdb

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

**The labels are:** 0 for negative sentiment and 1 for positive sentiment

In [3]:
train_dataset = imdb["train"]
test_dataset = imdb["test"]

### **First step: Preprocessing**

In [4]:
# Step 1:Create the datasets
import torch
from torch.utils.data.dataset import random_split

torch.manual_seed(42)
train_dataset, valid_dataset = random_split(train_dataset, [20000, 5000])


In [19]:
# Let's visulaize an example of data point:
train_dataset[0]

{'text': 'I saW this film while at Birmingham Southern College in 1975, when it was shown in combination with the Red Balloon. Both films are similar in their dream-like quality. The bulk of the film entails a fish swimming happily in his bowl while his new owner, a little boy, is away at school. A cat enters the room where the fish and his bowl are, and begins to warily stalk his "prey." The boy begins his walk home from school, and the viewer wonders whether he will arrive in time to save his fish friend. The fish becomes agitated by the cat\'s presence, and finally jumps out of the bowl! The cat quickly walks over to the fish, gently picks him up with his paws, and returns him to his bowl. The boy returns happily to his fish, none the wiser.<br /><br />The ending is amazing in both its irony and its technical complexity. It is hard to imagine how the director could\'ve pulled the technical feat back in 1959 -- it seems more a trick for 2003.<br /><br />If you can find it, watch it -

In [5]:
# Step 2: find unique tokens (words)
import re
from collections import Counter

def tokenizer(text):
    text = re.sub(r'<[^>]*>', '', text)

    emoticons = re.findall(
        r'(?::|;|=)(?:-)?(?:\)|\(|D|P)',
        text.lower()
    )

    text = re.sub(r'[\W]+', ' ', text.lower()) + \
           ' '.join(emoticons).replace('-', '')

    return text.split()


token_counts = Counter()

for item in train_dataset:
    tokens = tokenizer(item["text"])
    token_counts.update(tokens)

print("Vocab-size:", len(token_counts))

Vocab-size: 69209


In [6]:
!pip install -q torchtext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 1.4 MB/s eta 0:00:00


In [14]:
# Step 3: encoding each unique token into integers
token_to_idx={"<pad>": 0, "<unk>": 1}
token_to_idx.update({token[0]: idx for idx, token in enumerate(token_counts.most_common(), start=2)})

In [15]:
# example test
print([token_to_idx[token] for token in ['this', 'is', 'an', 'example']])

[11, 7, 35, 460]


In [26]:
# Step 3-A: define the functions for transformation
text_pipeline = lambda txt: [token_to_idx.get(token, token_to_idx["<unk>"]) for token in tokenizer(txt)]
label_pipeline= lambda lbl: 1. if lbl=='pos' else 0.

In [40]:
# Step 3-B: wrap the encode and transformation function
import torch
from torch import nn
def collate_batch(batch):
  label_list, text_list, lengths = [], [], []
  for item in batch:
    text = item["text"]
    label = item["label"]
    label_list.append(label_pipeline(label))
    processed_text= torch.tensor(text_pipeline(text), dtype=torch.int64)
    text_list.append(processed_text)
    lengths.append(processed_text.size(0))

  label_list= torch.tensor(label_list)
  lengths = torch.tensor(lengths)
  padded_text_list = nn.utils.rnn.pad_sequence(text_list, batch_first=True)

  return padded_text_list, label_list, lengths

# Take a small batch
from torch.utils.data import DataLoader
dataloader= DataLoader(train_dataset, batch_size=4,
                       shuffle=False, collate_fn= collate_batch)
next(iter(dataloader))

(tensor([[   10,   218,    11,  ...,     0,     0,     0],
         [ 7067,    31,    10,  ...,   117,  3042, 43116],
         [  260,    64,   110,  ...,     0,     0,     0],
         [   10,   140,   110,  ...,     0,     0,     0]]),
 tensor([0., 0., 0., 0.]),
 tensor([207, 261, 204, 132]))